# Adaptive Steane SE: circuits → Guppy/HUGR → LER

A compact example for **[[32,20,4]]**. Run the cells in order; optimization, fault analysis and compilation take a few minutes.

Setup (Python 3.12+, from the repository root): `python -m pip install -e ".[notebooks,guppy]"`. Select that environment as the notebook kernel.

After updating SpiderWarp or Guppy, **restart the kernel and run all cells** to load the current code. The setup cell prints the Python environment and Guppy version.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

import stim
from hugr.package import Package
from IPython.display import HTML, Markdown, display

from spiderwarp.csscode import CSSCode
from spiderwarp.utils import get_project_root
from spiderwarp.optimized_steane_se import optimized_steane_SE_orchestrator
from spiderwarp.se_protocol_simulation import estimate_protocol_ler

print(f"Python: {sys.executable}")
print(f"Guppy: {version('guppylang')}")
from spiderwarp.noise_model import CustomErrorModel, default_noise_model
from spiderwarp.se_protocol_inspection import circuit_inspection_html, circuit_resources, protocol_circuit_roles

from spiderwarp.stim_utils import circuit_for_inspection


Python: /opt/homebrew/Caskroom/miniforge/base/envs/error-correction/bin/python
Guppy: 0.21.6


**Specify the code and circuit distances.** `code.d` declares the code distance; the dictionary keys declare each supplied gadget's circuit distance. All input gadgets extract **X stabilizers**. The builder constructs their Z duals and derives the primary distance and fault budgets from `code.d`.

For another code, load its matrices and distance through `CSSCode`, and supply its distance-labelled circuits in the same form. The current builder supports identical `H_x`/`H_z` and code distances 2–5.

In [2]:
code_id = "32_20_4"
code = CSSCode.load_code("misc", code_id)  # Includes n, k, d, H_x/H_z and L_x/L_z.
assets = get_project_root() / "assets/circuits/SECircuits"
circuits = {
    d: stim.Circuit.from_file(assets / f"{code_id}_d{d}.stim")
    for d in (1, 2, 4)
}
print(f"Code [[{code.n},{code.k},{code.d}]]; circuit distances: {list(circuits)}")

Code [[32,20,4]]; circuit distances: [1, 2, 4]


**Inspect every supplied circuit.** Expand each view for a full-size, scrollable diagram and its annotated Stim instructions. Data wires are `q0` through `q31`; all remaining wires are ancillas. The optimized views below show the circuits actually executed by the protocol.

In [4]:
for distance, circuit in sorted(circuits.items()):
    display(HTML(circuit_inspection_html(circuit, f"Input X extraction, distance {distance}", code.n)))

**Optimize and export.** The clean cycle runs the maximum-distance Z primary, corrects, then runs the maximum-distance X primary and corrects. A raised primary flag discards that primary syndrome and selects a dual-basis recovery; where required, an additional fresh extraction resolves ambiguity. Raised recovery flags or ambiguous outcomes reject the shot.

Verification is **off by default**. Fault analysis still constructs the branch decisions. To check the resulting policy separately, call `protocol.verify()` (equivalently `verify_se_protocol(protocol)`); or pass `verify=True` during construction. The optional check uses Gottesman's conditions with separate CSS X/Z error weights. The standalone `.py` and `.hugr` contain the control flow and corrections, with no decoder sidecar.

In [7]:
output_dir = get_project_root().parent / "build" / f"{code_id}_protocol_demo"
recovery_objective = "qubits"  # "gates" favors shorter recoveries; compare both rates below.
protocol = optimized_steane_SE_orchestrator(code, circuits, output_dir=output_dir,
                                           recovery_objective=recovery_objective)
# Optional: report = protocol.verify()
print(f"Verification performed: {protocol.verification.verified}; peak SE qubits: {protocol.required_qubits}")
print(f"Correctable faults: {protocol.verification.correctable_faults}; "
      f"analyzed fault order: {protocol.verification.detectable_faults}")
print(f"Guppy: {protocol.source_path}")
print(f"HUGR:  {protocol.package_path}")

Verification performed: False; peak SE qubits: 60
Correctable faults: 1; analyzed fault order: 2
Guppy: /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/steane_se_protocol.py
HUGR:  /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/steane_se_protocol.hugr


**Inspect the circuits selected by the policy.** The table shows actual primary, recovery, and cleanup roles. An available circuit with no role is retained by the builder but is never selected by this policy. Gate counts include the Guppy basis rotations. Depth and live idle units use the same unit-duration scheduling model as the noise simulation, so they can help explain memory-noise exposure.

The complete annotated circuits and diagrams are saved beside the Guppy export. These views show the quantum gates; expand the Stim instructions to inspect the flag and syndrome parities.

In [8]:
inspection_dir = output_dir / "circuits"
inspection_dir.mkdir(parents=True, exist_ok=True)
roles = protocol_circuit_roles(protocol)
rows = ["| Circuit | Selected roles | Qubits | 2Q gates | 1Q gates | Depth | Live idle units |",
        "|---|---|---:|---:|---:|---:|---:|"]
for (basis, distance), gadget in sorted(protocol.gadgets.items()):
    r = circuit_resources(gadget.circuit, code.n)
    role = ", ".join(roles[basis, distance]) or "available, unused"
    rows.append(f"| {basis}, d{distance} | {role} | {r['qubits']} | {r['two_qubit_gates']} | "
                f"{r['one_qubit_gates']} | {r['depth']} | {r['live_idle_units']} |")
display(Markdown("\n".join(rows)))
for distance, circuit in sorted(circuits.items()):
    (inspection_dir / f"input_x_d{distance}.stim").write_text(str(circuit))
for (basis, distance), gadget in sorted(protocol.gadgets.items()):
    stem = f"{basis.lower()}_d{distance}"
    (inspection_dir / f"{stem}.stim").write_text(str(gadget.circuit))
    (inspection_dir / f"{stem}.svg").write_text(str(circuit_for_inspection(gadget.circuit).diagram("timeline-svg")))
    role = ", ".join(roles[basis, distance]) or "available, unused"
    display(HTML(circuit_inspection_html(gadget.circuit, f"Protocol {basis} extraction, distance {distance}: {role}", code.n)))
print(f"Saved circuit files: {inspection_dir}")

| Circuit | Selected roles | Qubits | 2Q gates | 1Q gates | Depth | Live idle units |
|---|---|---:|---:|---:|---:|---:|
| X, d1 | available, unused | 57 | 88 | 38 | 52 | 1144 |
| X, d2 | cleanup, recovery | 54 | 126 | 48 | 76 | 1904 |
| X, d4 | primary | 60 | 158 | 56 | 94 | 2516 |
| Z, d1 | available, unused | 57 | 88 | 102 | 68 | 1536 |
| Z, d2 | cleanup, recovery | 54 | 126 | 124 | 94 | 2374 |
| Z, d4 | primary | 60 | 158 | 142 | 116 | 3122 |

Saved circuit files: /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/circuits


**Load the exported HUGR.** This is a composable function package. To compose the Guppy source with a larger program, import its `se_cycle(data) -> bool` and honor the acceptance bit; `postselected_se_cycle(data)` instead terminates rejected shots. The generated module provides `compile_se_cycle()`: it selects minimal optimization when supported, or direct compilation with Guppy 0.21.6.

In [9]:
loaded_hugr = Package.from_bytes(Path(protocol.package_path).read_bytes())
guppy_cycle = protocol.guppy_function  # Callable from other @guppy functions.
print(f"Reloaded HUGR package with {len(loaded_hugr.modules)} module(s).")

Reloaded HUGR package with 1 module(s).


**Estimate one-cycle block LER.** Choose `prepared_state = "0"` or `"+"` below. All 20 logical qubits start in that state. Logical 0 inputs test logical X/Y errors; logical + inputs test logical Z/Y errors. A shot fails if any logical qubit has the wrong readout after ideal final `LutDecoder` correction.

This runs the adaptive **Stim reference interpreter** for the exported policy. Input preparation and final decoding are ideal. Each executed extraction and correction block is made noisy by `noise_model.py`'s `apply_noise`, using its **default hardware model**, including scheduled memory noise. Classical branch latency and later HUGR backend transformations are not modeled. To override the default, pass `noise_model=CustomErrorModel(...)` from `spiderwarp.noise_model`.

**LER is conditional on acceptance**; discards and a 95% Wilson interval are reported separately. Verification is optional for sampling too.

In [10]:
prepared_state = "0"  # Choose "0" or "+".
shots = 100_000
seed = 20261015
estimate = estimate_protocol_ler(protocol, prepared_state=prepared_state, shots=shots, seed=seed)
print(f"Prepared logical state: {prepared_state}; Stim {stim.__version__}")
print("Default noise parameters:", vars(estimate.noise_model))
lo, hi = estimate.ler_interval_95
rows = ["| Accepted / shots | Discard rate | Logical failures | Block LER | 95% interval |",
        "|---:|---:|---:|---:|---:|",
        f"| {estimate.accepted} / {estimate.shots} | {1 - estimate.acceptance_rate:.2%} | "
        f"{estimate.logical_errors} | {estimate.ler:.3%} | [{lo:.3%}, {hi:.3%}] |"]
display(Markdown("\n".join(rows)))

Prepared logical state: 0; Stim 1.15.0
Default noise parameters: {'p_1q': 2.5e-05, 'p_2q': 0.0008, 'p_meas': 1e-06, 'p_init': 0.0008, 'p_linear_mem': 1.2600000000000001e-05, 'p_coherent_mem': 0, 'p_leakage': 0.0}


| Accepted / shots | Discard rate | Logical failures | Block LER | 95% interval |
|---:|---:|---:|---:|---:|
| 95051 / 100000 | 4.95% | 869 | 0.914% | [0.856%, 0.977%] |

**Locate the discarded shots.** A primary flag starts recovery and can still lead to acceptance. The rejection table counts the terminal reason for each discarded shot exactly once. Percentages in that table use all shots; the stage table uses the number of times that particular circuit actually ran.

In [11]:
diagnostics = estimate.diagnostics
rows = ["| Last extraction | Rejection reason | Shots | % of all shots |",
        "|---|---|---:|---:|"]
for (role, basis, distance, reason), count in sorted(diagnostics.rejections.items(), key=lambda item: -item[1]):
    rows.append(f"| {role}: {basis}, d{distance} | {reason} | {count} | {count / shots:.2%} |")
display(Markdown("\n".join(rows)))
rows = ["| Executed extraction | Executions | Raised flags | Flag rate when executed |",
        "|---|---:|---:|---:|"]
for stage, count in diagnostics.stage_counts.items():
    role, basis, distance = stage
    raised = diagnostics.flagged_stage_counts.get(stage, 0)
    rows.append(f"| {role}: {basis}, d{distance} | {count} | {raised} | {raised / count:.2%} |")
display(Markdown("\n".join(rows)))
assert sum(diagnostics.rejection_counts.values()) == estimate.discarded

| Last extraction | Rejection reason | Shots | % of all shots |
|---|---|---:|---:|
| recovery: X, d2 | raised recovery flags | 1404 | 1.40% |
| recovery_cleanup: Z, d2 | raised recovery cleanup flags | 1298 | 1.30% |
| recovery: Z, d2 | raised recovery flags | 1094 | 1.09% |
| recovery_cleanup: X, d2 | raised recovery cleanup flags | 864 | 0.86% |
| primary: X, d4 | uncorrectable primary syndrome | 118 | 0.12% |
| recovery: Z, d2 | ambiguous recovery syndrome | 68 | 0.07% |
| recovery: X, d2 | ambiguous recovery syndrome | 42 | 0.04% |
| recovery_cleanup: X, d2 | ambiguous recovery syndrome | 37 | 0.04% |
| primary: Z, d4 | uncorrectable primary syndrome | 14 | 0.01% |
| primary: X, d4 | unmodeled primary flags | 10 | 0.01% |

| Executed extraction | Executions | Raised flags | Flag rate when executed |
|---|---:|---:|---:|
| primary: Z, d4 | 100000 | 14208 | 14.21% |
| primary: X, d4 | 85778 | 10775 | 12.56% |
| recovery: X, d2 | 14208 | 1404 | 9.88% |
| recovery_cleanup: Z, d2 | 12241 | 1298 | 10.60% |
| recovery: Z, d2 | 10765 | 1094 | 10.16% |
| recovery_cleanup: X, d2 | 9140 | 864 | 9.45% |

**Compare recovery choices on the same circuits.** The default `"qubits"` objective prefers the narrowest recovery, even when the primary already requires more qubits. The `"gates"` objective prefers fewer physical operations, considering recovery and cleanup together. Both retain the same fault-tolerance conditions during synthesis.

A shorter, unflagged recovery can accept more higher-order fault histories and therefore increase the conditional logical error rate. This comparison reports **both** rates before choosing a default. It rebuilds the classical decisions and Guppy export on the exact same optimized circuits, so it takes additional synthesis/compilation time. Independent verification remains optional: pass `verify=True` to `with_recovery_objective`, or call `alternative.verify()` separately.

In [12]:
comparison_shots = 20_000
other_objective = "gates" if protocol.recovery_objective == "qubits" else "qubits"
alternative = protocol.with_recovery_objective(other_objective, output_dir=output_dir / other_objective)
alternative_estimate = estimate_protocol_ler(alternative, prepared_state=prepared_state,
                                             shots=comparison_shots, seed=seed)
rows = ["| Recovery objective | Shots | Discard rate | 95% discard interval | Conditional block LER | 95% LER interval |",
        "|---|---:|---:|---:|---:|---:|"]
for objective, result in [(protocol.recovery_objective, estimate), (alternative.recovery_objective, alternative_estimate)]:
    dl, dh = result.discard_interval_95
    ll, lh = result.ler_interval_95
    rows.append(f"| {objective} | {result.shots} | {result.discard_rate:.2%} | [{dl:.2%}, {dh:.2%}] | "
                f"{result.ler:.3%} | [{ll:.3%}, {lh:.3%}] |")
display(Markdown("\n".join(rows)))
print("Peak available SE qubits:", protocol.required_qubits, "(same for both policies)")

| Recovery objective | Shots | Discard rate | 95% discard interval | Conditional block LER | 95% LER interval |
|---|---:|---:|---:|---:|---:|
| qubits | 100000 | 4.95% | [4.82%, 5.09%] | 0.914% | [0.856%, 0.977%] |
| gates | 20000 | 0.39% | [0.31%, 0.49%] | 1.978% | [1.793%, 2.181%] |

Peak available SE qubits: 60 (same for both policies)


**Separate noise sources on this same protocol.** This smaller comparison keeps the circuits and recovery policy fixed. Removing or scaling a noise source is a diagnostic experiment; it does not change the main simulation's default hardware model. Noise contributions interact, so the percentages are not additive. Conditional LER is shown alongside discard rate to expose an acceptance/accuracy tradeoff.

In [13]:
diagnostic_shots = 10_000
base = vars(default_noise_model())
noise_cases = {
    "Noiseless": CustomErrorModel(),
    "Default hardware model": CustomErrorModel(**base),
    "No memory noise": CustomErrorModel(**dict(base, p_linear_mem=0, p_coherent_mem=0, p_leakage=0)),
    "Memory noise only": CustomErrorModel(p_linear_mem=base["p_linear_mem"], p_coherent_mem=base["p_coherent_mem"], p_leakage=base["p_leakage"]),
    "Two-qubit noise only": CustomErrorModel(p_2q=base["p_2q"]),
    "Quarter memory rate (sensitivity)": CustomErrorModel(**dict(base, p_linear_mem=base["p_linear_mem"] / 4)),
}
noise_estimates = {}
for label, model in noise_cases.items():
    noise_estimates[label] = estimate_protocol_ler(protocol, noise_model=model, prepared_state=prepared_state,
                                                   shots=diagnostic_shots, seed=seed)
rows = ["| Noise case | Discard rate | 95% discard interval | Conditional block LER |",
        "|---|---:|---:|---:|"]
for label, result in noise_estimates.items():
    low, high = result.discard_interval_95
    rows.append(f"| {label} | {result.discard_rate:.2%} | [{low:.2%}, {high:.2%}] | {result.ler:.3%} |")
display(Markdown("\n".join(rows)))
assert noise_estimates["Noiseless"].discarded == noise_estimates["Noiseless"].logical_errors == 0


| Noise case | Discard rate | 95% discard interval | Conditional block LER |
|---|---:|---:|---:|
| Noiseless | 0.00% | [0.00%, 0.04%] | 0.000% |
| Default hardware model | 4.86% | [4.46%, 5.30%] | 0.799% |
| No memory noise | 4.20% | [3.82%, 4.61%] | 0.877% |
| Memory noise only | 0.11% | [0.06%, 0.20%] | 0.000% |
| Two-qubit noise only | 2.05% | [1.79%, 2.35%] | 0.582% |
| Quarter memory rate (sensitivity) | 4.26% | [3.88%, 4.67%] | 0.742% |

**Check the other logical basis and save the diagnostics.** The default memory channel is Z-biased, so logical `0` alone can hide phase errors. The smaller complementary-basis check reports its own conditional LER. The JSON report records the circuits, noise parameters, rejection counts, and sampling settings for comparison across optimization runs.

In [14]:
other_basis = "+" if prepared_state == "0" else "0"
other_estimate = estimate_protocol_ler(protocol, prepared_state=other_basis, shots=diagnostic_shots, seed=seed)
rows = ["| Input basis | Shots | Discard rate | Conditional block LER | 95% LER interval |",
        "|---|---:|---:|---:|---:|"]
for result in (estimate, other_estimate):
    low, high = result.ler_interval_95
    rows.append(f"| {result.prepared_state} | {result.shots} | {result.discard_rate:.2%} | {result.ler:.3%} | [{low:.3%}, {high:.3%}] |")
display(Markdown("\n".join(rows)))

import hashlib
import json

def sampling_summary(result):
    return dict(shots=result.shots, accepted=result.accepted, logical_errors=result.logical_errors,
                prepared_state=result.prepared_state, seed=result.seed, noise=vars(result.noise_model),
                discard_rate=result.discard_rate, discard_interval_95=result.discard_interval_95,
                conditional_ler=result.ler, ler_interval_95=result.ler_interval_95)

report = dict(code=code_id, stim_version=stim.__version__, guppy_version=version("guppylang"),
              recovery_objective=protocol.recovery_objective,
              alternative_objective=alternative.recovery_objective, alternative=sampling_summary(alternative_estimate),
              primary=sampling_summary(estimate), complementary=sampling_summary(other_estimate),
              noise_cases={label: sampling_summary(result) for label, result in noise_estimates.items()},
              rejections=[dict(role=role, basis=basis, distance=distance, reason=reason, count=count)
                          for (role, basis, distance, reason), count in diagnostics.rejections.items()],
              circuits=[dict(basis=basis, distance=distance, roles=roles[basis, distance],
                             sha256=hashlib.sha256(str(gadget.circuit).encode()).hexdigest(),
                             resources=circuit_resources(gadget.circuit, code.n))
                        for (basis, distance), gadget in sorted(protocol.gadgets.items())])
report_path = output_dir / "simulation_diagnostics.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"Saved sampling diagnostics: {report_path}")

| Input basis | Shots | Discard rate | Conditional block LER | 95% LER interval |
|---|---:|---:|---:|---:|
| 0 | 100000 | 4.95% | 0.914% | [0.856%, 0.977%] |
| + | 10000 | 4.86% | 1.429% | [1.210%, 1.688%] |

Saved sampling diagnostics: /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/simulation_diagnostics.json


Zero observed failures give an upper uncertainty limit, not proof that LER is zero. This estimates one SE cycle from clean logical 0/+ input, conditional on acceptance. Sampling includes all fault orders; optional exact verification uses the bounded circuit-level Pauli model described in its report. Seeds are reproducible with a fixed Stim version and machine architecture.